# ProGen3 — Protein Sequence Generation & Scoring

ProGen3 is a Mixture-of-Experts protein language model from Profluent AI supporting:
- **Forward (N→C) generation**: set `direction="forward"` (default)
- **Reverse (C→N) generation**: set `direction="reverse"`
- **Bidirectional scoring**: averages forward + reverse log-likelihoods

Available checkpoints: `progen3-112m`, `progen3-219m`, `progen3-339m`, `progen3-762m`, `progen3-1b`, `progen3-3b`

GitHub: https://github.com/Profluent-AI/progen3

In [1]:
from proto_tools.tools.causal_models.progen3 import (
    ProGen3SampleConfig,
    ProGen3SampleInput,
    ProGen3ScoringConfig,
    ProGen3ScoringInput,
    run_progen3_sample,
    run_progen3_score,
)

## Forward Generation (N→C)

Forward generation extends the sequence from the N-terminus (default direction).

In [2]:
# Forward generation from an N-terminal prompt
inputs = ProGen3SampleInput(prompts=["MKTLVIVTGA"])
config = ProGen3SampleConfig(
    model_checkpoint="progen3-762m",
    temperature=0.2,
    top_p=0.95,
    max_new_tokens=100,
)
result = run_progen3_sample(inputs, config)

print(f"Success: {result.success}")
print(f"Generated {len(result.sequences)} sequence(s)")
print(f"Sequence: {result.sequences[0][:80]}...")

Success: True
Generated 1 sequence(s)
Sequence: MKTLVIVTGASSGIGRATARLFAQRGYRVFNLSRRACPVEGVTPLACDVTDEAAVRAAVAEVLARAGRIDVLVNNAGFGV...


## Reverse Generation (C→N)

Set `direction="reverse"` to generate from the C-terminus.

In [3]:
# Reverse generation from a C-terminal prompt
inputs = ProGen3SampleInput(prompts=["RYTEAFLK"])
config = ProGen3SampleConfig(
    model_checkpoint="progen3-762m",
    direction="reverse",
    max_new_tokens=100,
)
result = run_progen3_sample(inputs, config)

print(f"Generated: {result.sequences[0][:80]}...")

Generated: PHGLANALLLPHVMEFNLPAAPERFARIARLLGADVAGLSDEEAAERAVAAVEALAERVGLPRRLRDLGVPEADIPALAE...


## Unconditional Generation

Pass an empty string prompt for unconditional generation.

In [4]:
# Unconditional forward generation
inputs = ProGen3SampleInput(prompts=[""])
config = ProGen3SampleConfig(max_new_tokens=200)
result = run_progen3_sample(inputs, config)

print(f"Generated: {result.sequences[0][:80]}...")
print(f"Length: {len(result.sequences[0])} residues")

Generated: MKKILLLLLIPLLLFSCSKDDDNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
Length: 202 residues


## Scoring

In [5]:
# Score protein sequences
inputs = ProGen3ScoringInput(
    sequences=[
        "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF",  # Hemoglobin alpha fragment
        "MKTLLILAVVAAALA",  # Signal peptide
    ]
)
config = ProGen3ScoringConfig(model_checkpoint="progen3-762m")
result = run_progen3_score(inputs, config)

print(f"Success: {result.success}")
for i, score in enumerate(result.scores):
    print(f"\nSequence {i}:")
    print(f"  Log-likelihood: {score.metrics['log_likelihood']:.4f}")
    print(f"  Avg log-likelihood: {score.metrics['avg_log_likelihood']:.4f}")
    print(f"  Perplexity: {score.metrics['perplexity']:.4f}")

Success: True

Sequence 0:
  Log-likelihood: -2.6666
  Avg log-likelihood: -2.6666
  Perplexity: 14.3903

Sequence 1:
  Log-likelihood: -2.2830
  Avg log-likelihood: -2.2830
  Perplexity: 9.8064


## Export Results

In [6]:
# Export generated sequences to FASTA
inputs = ProGen3SampleInput(prompts=["MKTL"])
result = run_progen3_sample(inputs, ProGen3SampleConfig(max_new_tokens=50))
result.export("progen3_generated", export_path="./example_output", file_format="fasta")
print("Exported to ./example_output/progen3_generated.fasta")

# Export scoring results to CSV
inputs = ProGen3ScoringInput(sequences=["MVLSPADKTNVKAAW", "MKTLLILAVVAA"])
score_result = run_progen3_score(inputs)
score_result.export("progen3_scores", export_path="./example_output", file_format="csv")
print("Exported to ./example_output/progen3_scores.csv")

Exported to ./example_output/progen3_generated.fasta
Exported to ./example_output/progen3_scores.csv
